# 07 · Comprobación y vínculo con el informe

En [00](00_inicio_y_referencia.ipynb) identifiqué las hipótesis y la ejecución; en 01–06 preparé demanda, recorridos, vehículos, servicio, factores COPERT y balance. Ahora compruebo **si sus productos reproducen la referencia** y explico dónde aparecen sus resultados en el informe y los anexos. Mantengo **2.500.000 visitas-persona anuales sin y con intervención**, las 31 condiciones y la programación vigente.

**Recibe:** los productos aceptados de 01–06 de la misma ejecución, sus referencias de comparación y las matrices documentales. **Produce al comparar:** 170 comparaciones identificadas en el registro, con diferencias y tolerancias; no vuelve a calcular demanda, factores o energía. **Permite localizar:** 16 productos técnicos, 33 anexos de datos, 392 definiciones de columnas, 43 tablas, 20 figuras y ocho fichas metodológicas de la entrega.

La referencia documental es el **informe v0.28, edición del 16/11/2026**. Aquí consulto sus correspondencias y los documentos disponibles según la política de publicación: no compilo otra edición, no genero recursos editoriales y no modifico informe, anexos o Gold. La verificación numérica está en [etapas_registradas.py](../src/movilidad_teide/flujos/etapas_registradas.py); la documental, en [verificar_trazabilidad.py](../scripts/verificar_trazabilidad.py). El cuaderno utiliza esas funciones comunes.


## 1. Consultar una comparación aceptada o realizarla

La opción inicial **`consultar`** lee 07 ya completada y comprueba la integridad y los vínculos disponibles. **`comparar`** ejecuta 07 sobre los productos de 01–06 de esa misma carpeta. Si falta una etapa, se detiene: no utiliza datos de otra ejecución para completarla. Una comparación completada se conserva; para repetirla inicio otra ejecución o continúo una que aún tenga 07 pendiente.

El origen de los factores de 05 se muestra por separado. Pueden ser distribuidos o proceder de un motor nuevo: **comparar resultados no lanza COPERT**. La acción elegida aquí tampoco cambia hipótesis ni busca otro servicio de transporte.

Los controles de recurso y anexo solo seleccionan qué documento inspeccionar; no cambian el cálculo. Por defecto sigo la tabla 24 del informe y su anexo de balance. Los CSV completos permiten elegir cualquier otra tabla, figura o anexo de la misma entrega.


In [1]:
from pathlib import Path
import sys,json,hashlib,runpy,html,re
from urllib.parse import quote
import pandas as pd
import numpy as np
from IPython.display import display,Markdown
candidatas=[Path.cwd().resolve(),*Path.cwd().resolve().parents]
RAIZ=next((p for p in candidatas if (p/"pyproject.toml").is_file()
           and (p/"config/parametros_modelo.xlsx").is_file()),None)
if RAIZ is None:
    raise FileNotFoundError("Abra este cuaderno desde la raíz del proyecto o desde notebooks/.")
sys.path.insert(0,str(RAIZ/"src"))
from movilidad_teide.configuracion.registro import abrir_ejecucion,resumen_ejecucion
from movilidad_teide.flujos.etapas_registradas import ejecutar_etapa


In [2]:
ACCION_COMPROBACION="consultar"  # Alternativa explícita: "comparar".
RUTA_EJECUCION=Path("data/silver/ejecuciones/referencia_2028_final_20261116")
TIPO_RECURSO="Tabla"  # Selección documental: "Tabla" o "Figura".
NUMERO_RECURSO=24
ANEXO_LECTURA="balance_anual.csv"


In [3]:
if ACCION_COMPROBACION not in {"consultar","comparar"}:
    raise ValueError('Elija consultar o comparar en ACCION_COMPROBACION.')
EJECUCION=RAIZ/RUTA_EJECUCION
if not (EJECUCION/"registro_ejecucion.json").is_file():
    raise FileNotFoundError("No existe el registro elegido. Inicie 00 y complete sus etapas con la misma ruta.")
DATOS=abrir_ejecucion(RAIZ,EJECUCION,comprobar_referencias=True)
if ACCION_COMPROBACION=="comparar":
    faltan=[s for s in ["01","02","03","04","05","06"] if DATOS["estado_etapas"]["etapas"][s]["estado"]!="completada"]
    if faltan:
        raise ValueError("Complete primero las etapas de esta misma ejecución: "+", ".join(faltan))
    if DATOS["estado_etapas"]["etapas"]["07"]["estado"]=="completada":
        raise ValueError("07 ya está completada. No se sobrescriben comparaciones aceptadas.")
    ejecutar_etapa(RAIZ,EJECUCION,"07",accion="comparar")
    DATOS=abrir_ejecucion(RAIZ,EJECUCION,comprobar_referencias=True)
ETAPA=DATOS["estado_etapas"]["etapas"]["07"]
if ETAPA["estado"]!="completada":
    raise ValueError("07 no está completada. Prepare 01–06 y elija comparar en esta ejecución.")
COMPROBACIONES=EJECUCION/ETAPA["carpeta"]
VERIFICACION=json.loads((COMPROBACIONES/"verificacion_integrada.json").read_text())
RESUMEN=resumen_ejecucion(DATOS)
CFG=DATOS["configuracion"]["valores"]
assert VERIFICACION["equivalente"] and not VERIFICACION["informe_y_anexos_modificados"]
print("Acción realizada:",ACCION_COMPROBACION)
print("Ejecución:",RESUMEN["identificador"])
print("Visitas-persona sin y con intervención:",RESUMEN["visitas_sin_y_con"])
print("Condiciones efectivas:",RESUMEN["condiciones_vigentes"])
print("Origen de los factores de 05:",DATOS["estado_etapas"]["etapas"]["05"]["accion"])
print("Motor COPERT ejecutado en 07: no; se comparan productos ya calculados.")
print("Carpeta de comprobaciones dentro del registro:",ETAPA["carpeta"])
def mostrar(tabla,titulo,formatos=None):
    display(tabla.style.set_uuid("comprobacion07").hide(axis="index").set_caption(titulo)
            .format(formatos or {},decimal=",",thousands=" ",na_rep="No procede"))
def leer_matriz(nombre):
    return pd.read_csv(RAIZ/"manifiestos"/(nombre+".csv"),sep=";",keep_default_na=False)
from movilidad_teide.publicacion.reserva import leer_politica
POLITICA_DOCUMENTAL=leer_politica(RAIZ)
def enlace(nombre,etiqueta=None,fragmento=""):
    if not (RAIZ/nombre).exists() and not POLITICA_DOCUMENTAL.excluye(nombre):
        raise FileNotFoundError("No existe el archivo documental: "+nombre)
    return '<a href="../'+quote(nombre,safe="/")+fragmento+'">'+html.escape(etiqueta or nombre)+'</a>'
PDF="docs/informe_energetico/INFORME_ENERGETICO_v0.28_2026-11-16.pdf"


Acción realizada: consultar
Ejecución: referencia_2028_paquetes_20261116
Visitas-persona sin y con intervención: 2500000
Condiciones efectivas: 31
Origen de los factores de 05: consultar
Motor COPERT ejecutado en 07: no; se comparan productos ya calculados.
Carpeta de comprobaciones dentro del registro: resultados/07/fd4fccbd26284d99a05ee20f034f0df3


## 2. Qué demuestra cada comprobación

**Integridad** significa que los archivos, el código, la configuración y los productos coinciden con las huellas SHA-256 que los identifican. Una huella cambia al cambiar el contenido. **Equivalencia numérica** significa que los resultados respetan claves, ausencias, geometrías y tolerancias de sus contratos. **Trazabilidad documental** significa que existen las correspondencias entre resultado, fuente, código, salida y recurso del informe.

Son comprobaciones complementarias. La matriz documental no es una auditoría independiente de cada celda, y una coincidencia informática no confirma disponibilidad de autobuses, permisos, demanda real o aceptación institucional. Las decisiones técnicas fijadas y las hipótesis se mantienen identificadas como tales.

**Tabla 1. Estado de las etapas de esta ejecución del estudio del Teide, 2028.** Fuente: registro reabierto y verificado. Las carpetas son relativas a ese registro; los productos no son vehículos ni visitas. Las acciones de etapas anteriores indican qué se hizo al aceptarlas, no tareas repetidas ahora.


In [4]:
filas=[]
for numero,e in DATOS["estado_etapas"]["etapas"].items():
    filas.append({"Etapa":numero,"Materia":e["nombre"],"Estado":e["estado"],"Acción aceptada":e.get("accion",""),
                  "Productos identificados":len(e.get("productos",[])),"Carpeta dentro del registro":e.get("carpeta","")})
mostrar(pd.DataFrame(filas),"Tabla 1 · Etapas y procedencia aceptadas · Teide 2028 · Mismas2.500.000 visitas y 31 condiciones")
assert all(x["estado"]=="completada" for x in DATOS["estado_etapas"]["etapas"].values())


Etapa,Materia,Estado,Acción aceptada,Productos identificados,Carpeta dentro del registro
01,Fuentes y demanda,completada,calcular,54,resultados/01/bec978f433644f97a256a6878c6d8f1d
02,"Recorridos, pendientes y clasificación vial",completada,calcular,172,resultados/02/4218a4403c0d41ab9f631c6f05a54109
03,Representación técnica del parque,completada,registrar_datos_fijados,6,resultados/03/8794af6e180a4004b08da5cf5d639df3
04,Servicio de transporte y actividad,completada,comprobar,1 989,resultados/04/edb4658dccce43a19a8bb2b4496d72a6
05,Factores COPERT,completada,consultar,6,resultados/05/ab7b8a70e5ef40c795f02f08881c0219
06,"Coeficientes, balance y sensibilidades",completada,calcular,2 060,resultados/06/523b20eb58e44554ab7808d3358b08ed
07,Comprobación de equivalencia,completada,comparar,8,resultados/07/fd4fccbd26284d99a05ee20f034f0df3


## 3. Comparación numérica: demanda, geografía y balance

La tabla 2 reúne **170 comparaciones**: 21 de demanda, 116 geográficas y 33 de coeficientes/balance, incluidos 16 productos anuales. Se conservan las tolerancias originales; no se redondean datos ni se cambian valores esperados para obtener coincidencia. Las diferencias se expresan en las unidades de cada columna comparada, por lo que no se interpreta su máximo global como MWh, metros o visitas indistintamente.

En geografía se separa la diferencia numérica de la diferencia de coordenadas. El contrato de demanda compara claves y cantidades de las 328 filas territoriales; sus atributos adicionales de perfil se comprueban en las preparaciones geográficas correspondientes. Las decisiones de representación técnica de 03 y los programas de 04 siguen siendo elecciones fijadas, no resultados de una búsqueda nueva.

**Tabla 2. Equivalencia de los productos de la ejecución con la referencia, Teide 2028.** Fuente: `verificacion_integrada.json` y comparaciones de 07. La tolerancia numérica es absoluta y se aplica en las unidades de cada contrato; no constituye un margen de error observado del transporte real.


In [5]:
comparaciones=[]
for clave,nombre in [("demanda","Demanda"),("geografia","Geografía y perfiles"),("balance","Coeficientes y balance")]:
    parte=VERIFICACION[clave]
    tabla=pd.read_csv(COMPROBACIONES/clave/"comparacion_equivalencia.csv",sep=";")
    assert len(tabla)==parte["interfaces_comparadas"] and tabla.equivalente.all()
    comparaciones.append({"Grupo":nombre,"Comparaciones":parte["interfaces_comparadas"],
        "Equivalentes":"Sí" if parte["equivalente"] else "No",
        "Diferencia numérica máxima":parte.get("maxima_diferencia_absoluta",parte.get("maxima_diferencia_numerica_absoluta")),
        "Tolerancia absoluta":parte["tolerancia_absoluta"],
        "Diferencia de coordenadas (m)":parte.get("maxima_diferencia_coordenadas_m",np.nan)})
COMPARACION_NUMERICA=pd.DataFrame(comparaciones)
assert int(COMPARACION_NUMERICA.Comparaciones.sum())==170
mostrar(COMPARACION_NUMERICA,"Tabla 2 · Comparaciones numéricas y geográficas aceptadas · Teide 2028",{"Diferencia numérica máxima":"{:.3g}","Tolerancia absoluta":"{:.1g}","Diferencia de coordenadas (m)":"{:.3g}"})
print("Productos anuales incluidos en balance:",VERIFICACION["balance"]["productos_anuales"])


Grupo,Comparaciones,Equivalentes,Diferencia numérica máxima,Tolerancia absoluta,Diferencia de coordenadas (m)
Demanda,21,Sí,0,1e-08,No procede
Geografía y perfiles,116,Sí,"1,42e-14",1e-08,0
Coeficientes y balance,33,Sí,"1,46e-11",1e-08,No procede


Productos anuales incluidos en balance: 16


## 4. Integridad y cobertura de las correspondencias

La celda llama al mismo verificador que las instrucciones de reproducción. Comprueba inventario, referencias numéricas obligatorias, catálogo, funciones y correspondencias. La política de `.gitignore` determina además qué documentación debe estar disponible.

**Con reserva**, no se exige el informe, sus fuentes o los anexos de entrega. Se mantienen sus matrices de procedencia y unidades, sin afirmar una comprobación de archivos ausentes. **Sin reserva**, se exigen también maestro, recursos y cabeceras de anexos y se coteja su integridad. Un dato científico ausente o alterado falla en ambas modalidades.

**Tabla 3. Cobertura catalogada del estudio del Teide, 2028, y archivos comprobados en esta modalidad.** Fuente: matrices y verificador ejecutado ahora. Los 33 anexos, 392 columnas, 43 tablas, 20 figuras y ocho fichas describen el catálogo; la salida distingue expresamente si también se han comprobado sus documentos. No se generan archivos editoriales.


In [6]:
VERIFICAR_DOCUMENTOS=runpy.run_path(str(RAIZ/"scripts/verificar_trazabilidad.py"))["comprobar"]
INTEGRIDAD=VERIFICAR_DOCUMENTOS(RAIZ)
RECURSOS=leer_matriz("trazabilidad_recursos")
ANEXOS=leer_matriz("trazabilidad_anexos")
COLUMNAS=leer_matriz("trazabilidad_columnas")
PRODUCTOS=leer_matriz("trazabilidad_resultados")
FICHAS=leer_matriz("trazabilidad_fichas")
HIPOTESIS=leer_matriz("trazabilidad_hipotesis")
BIBLIOGRAFIA=leer_matriz("fuentes_bibliograficas")
MAPA=leer_matriz("mapa_recorrido_vinculos")
filas=[("Productos técnicos anuales",len(PRODUCTOS)),("Anexos de datos",len(ANEXOS)),("Definiciones de columnas",len(COLUMNAS)),
       ("Tablas numeradas",int(RECURSOS.Tipo.eq("Tabla").sum())),("Figuras numeradas",int(RECURSOS.Tipo.eq("Figura").sum())),
       ("Fichas metodológicas",len(FICHAS)),("Condiciones del libro",len(HIPOTESIS)),("Referencias bibliográficas",len(BIBLIOGRAFIA)),
       ("Archivos del inventario verificados",INTEGRIDAD["archivos_verificados"])]
mostrar(pd.DataFrame(filas,columns=["Elemento","Cantidad"]),"Tabla 3 · Cobertura e integridad comprobadas · Informe v0.28,16/11/2026 · Fuente: matrices y verificador")
print("Trazabilidad documental verificada:",INTEGRIDAD["trazabilidad_documental_verificada"])


Elemento,Cantidad
Productos técnicos anuales,16
Anexos de datos,33
Definiciones de columnas,392
Tablas numeradas,43
Figuras numeradas,20
Fichas metodológicas,8
Condiciones del libro,31
Referencias bibliográficas,22
Archivos del inventario verificados,11 216


Trazabilidad documental verificada: False


## 5. De la salida de esta ejecución a los anexos de lectura

La tabla 4 relaciona cada producto técnico anual de 06 con su referencia Gold y los anexos que permiten leerlo. La carpeta de 06 se resuelve desde este registro; nunca se elige una ejecución por el nombre más reciente. Las rutas relativas son iguales en desarrollo y publicación.

El archivo técnico `balance_anual_politicas.csv` contiene las cuentas calculadas. `data/gold/anexos/balance_anual.csv` es su presentación legible para el informe. La matriz explica la transformación y el productor editorial que los relaciona; son funciones distintas del mismo desarrollo. **Generar esa presentación no equivale a repetir el cálculo.** Aquí localizo los productos técnicos obligatorios y señalo la disponibilidad de los anexos de lectura, que pueden estar reservados.

**Tabla 4. Dieciséis productos anuales comparados y sus anexos relacionados, Teide 2028.** Fuente: matriz de resultados, índice de referencia y productos aceptados de 06/07. El resultado de equivalencia corresponde al producto técnico; no se atribuye por ello una auditoría independiente de todas las celdas maquetadas.


In [7]:
INDICE=DATOS["configuracion"]["indice"]
CARPETA_BALANCE=EJECUCION/DATOS["estado_etapas"]["etapas"]["06"]["carpeta"]
BALANCE_COMPARADO=pd.read_csv(COMPROBACIONES/"balance/comparacion_equivalencia.csv",sep=";")
filas=[]
for nombre,referencia in INDICE["productos_actualizados"].items():
    metadata=PRODUCTOS.loc[PRODUCTOS.Producto.eq(referencia)]
    if len(metadata)!=1:raise ValueError("Producto sin correspondencia documental única: "+nombre)
    salida="balance/"+nombre
    control=BALANCE_COMPARADO.loc[BALANCE_COMPARADO.producto.eq(salida)]
    if len(control)!=1 or not control.equivalente.all():raise ValueError("Producto no comparado: "+nombre)
    archivo=CARPETA_BALANCE/salida
    if not archivo.is_file():raise FileNotFoundError("Producto de 06ausente: "+nombre)
    anexos=metadata.iloc[0]["Anexos de lectura"].split(" | ")
    filas.append({"Producto técnico":nombre,"Ruta dentro del registro":archivo.relative_to(EJECUCION).as_posix(),
      "Referencia Gold":enlace(referencia),"Anexos relacionados":"<br>".join(enlace(x) for x in anexos if x),"Equivalente":"Sí"})
VINCULOS_PRODUCTOS=pd.DataFrame(filas)
assert len(VINCULOS_PRODUCTOS)==16
mostrar(VINCULOS_PRODUCTOS,"Tabla 4 · Productos de 06, referencia y anexos de lectura · Teide 2028 · Equivalencia verificada por 07")


Producto técnico,Ruta dentro del registro,Referencia Gold,Anexos relacionados,Equivalente
energia_por_programa.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/energia_por_programa.csv,data/gold/referencia_anual/energia_por_programa.csv,data/gold/anexos/balance_anual.csvdata/gold/anexos/energia_componentes.csv,Sí
balance_anual_politicas.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/balance_anual_politicas.csv,data/gold/referencia_anual/balance_anual_politicas.csv,data/gold/anexos/balance_anual.csvdata/gold/anexos/balance_regimenes.csvdata/gold/anexos/energia_diaria.csv,Sí
servicios_anuales_politicas.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/servicios_anuales_politicas.csv,data/gold/referencia_anual/servicios_anuales_politicas.csv,data/gold/anexos/balance_anual.csvdata/gold/anexos/balance_regimenes.csvdata/gold/anexos/energia_diaria.csv,Sí
resumen_politicas.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/resumen_politicas.csv,data/gold/referencia_anual/resumen_politicas.csv,data/gold/anexos/balance_anual.csvdata/gold/anexos/balance_regimenes.csvdata/gold/anexos/energia_diaria.csv,Sí
calendarios_politicas.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/calendarios_politicas.csv,data/gold/referencia_anual/calendarios_politicas.csv,data/gold/anexos/balance_anual.csvdata/gold/anexos/balance_regimenes.csvdata/gold/anexos/energia_diaria.csv,Sí
actividad_y_energia_diaria_politicas.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/actividad_y_energia_diaria_politicas.csv,data/gold/referencia_anual/actividad_y_energia_diaria_politicas.csv,data/gold/anexos/balance_anual.csvdata/gold/anexos/balance_regimenes.csvdata/gold/anexos/energia_diaria.csv,Sí
recomendada_calendario.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/recomendada_calendario.csv,data/gold/referencia_anual/recomendada_calendario.csv,data/gold/anexos/calendario_diario.csvdata/gold/anexos/programacion_anual.csvdata/gold/anexos/movimientos_lanzaderas.csvdata/gold/anexos/reservas_visita.csvdata/gold/anexos/disponibilidad_unidades.csv,Sí
recomendada_movimientos_calendario.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/recomendada_movimientos_calendario.csv,data/gold/referencia_anual/recomendada_movimientos_calendario.csv,data/gold/anexos/calendario_diario.csvdata/gold/anexos/programacion_anual.csvdata/gold/anexos/movimientos_lanzaderas.csvdata/gold/anexos/reservas_visita.csvdata/gold/anexos/disponibilidad_unidades.csv,Sí
recomendada_disponibilidad_calendario.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/recomendada_disponibilidad_calendario.csv,data/gold/referencia_anual/recomendada_disponibilidad_calendario.csv,data/gold/anexos/calendario_diario.csvdata/gold/anexos/programacion_anual.csvdata/gold/anexos/movimientos_lanzaderas.csvdata/gold/anexos/reservas_visita.csvdata/gold/anexos/disponibilidad_unidades.csv,Sí
recomendada_jornadas_calendario.csv,resultados/06/523b20eb58e44554ab7808d3358b08ed/balance/recomendada_jornadas_calendario.csv,data/gold/referencia_anual/recomendada_jornadas_calendario.csv,data/gold/anexos/calendario_diario.csvdata/gold/anexos/programacion_anual.csvdata/gold/anexos/movimientos_lanzaderas.csvdata/gold/anexos/reservas_visita.csvdata/gold/anexos/disponibilidad_unidades.csv,Sí


## 6. Seguir una tabla o figura del informe

Los controles iniciales seleccionan una tabla o figura por su número en la entrega, no por un código de escenario. La tabla 5 muestra su título, página del PDF, fuentes, entradas, código, anexos y límites. Las rutas son comunes a ambas copias. Los documentos distribuidos tienen enlace; los reservados muestran una explicación de disponibilidad.

Por defecto se inspecciona la **tabla 24, consumo y ahorro anual**, en la página que identifica la matriz del PDF entregado. La cadena empieza en las fuentes y el Excel, continúa por 01–06, se contrasta en 07 y llega al anexo y a su recurso maquetado. La figura 11 relaciona calendario y energía; las tablas 35–36 presentan la sensibilidad de excursiones. Sus números y páginas se toman de la matriz de esa misma edición.

**Tabla 5. Trazabilidad del recurso seleccionado del informe v0.28, 16/11/2026.** Fuente: matriz de recursos. «Código de cálculo» y «productor del recurso» cumplen funciones distintas; el cuaderno no ejecuta ese productor editorial. La página enlazada corresponde al PDF entregado, que se conserva sin cambios.


In [8]:
if TIPO_RECURSO not in {"Tabla","Figura"}:
    raise ValueError("TIPO_RECURSO debe ser Tabla o Figura.")
elegido=RECURSOS.loc[RECURSOS.Tipo.eq(TIPO_RECURSO)&RECURSOS["Número"].astype(int).eq(int(NUMERO_RECURSO))]
if len(elegido)!=1:raise ValueError("El recurso seleccionado no existe o no es único en la entrega.")
RECURSO=elegido.iloc[0]
pagina=int(RECURSO["Página PDF"])
campos=[("Recurso",html.escape(f"{TIPO_RECURSO} {NUMERO_RECURSO} · "+RECURSO["Título"])),
        ("PDF entregado y página",enlace(PDF,f"Informe v0.28 · página {pagina}",f"#page={pagina}")),
        ("Archivo del recurso",enlace(RECURSO["Recurso"])),
        ("Sección del informe",html.escape(RECURSO["Sección"])),
        ("Unidades",html.escape(RECURSO["Unidades"])),
        ("Fuentes e hipótesis",html.escape(RECURSO["Fuentes e hipótesis"])),
        ("Referencias citadas en el informe",html.escape(RECURSO["Fuentes citadas en informe"])),
        ("Entradas de aplicación documentadas",html.escape(RECURSO["Entradas de aplicación"])),
        ("Código de cálculo documentado",html.escape(RECURSO["Código de cálculo"])),
        ("Productor o reutilización del recurso",html.escape(RECURSO["Productor o reutilización"])),
        ("Anexos relacionados","<br>".join((enlace(x) if "/" in x or x.endswith((".csv",".parquet",".xlsx",".pdf")) else html.escape(x))
                                            for x in RECURSO["Anexos relacionados"].split(" | ") if x)),
        ("Tratamiento del recurso",html.escape(RECURSO["Tratamiento del recurso"])),
        ("Comprobación y alcance",html.escape(RECURSO["Comprobación"])),
        ("Límite",html.escape(RECURSO["Límite"]))]
mostrar(pd.DataFrame(campos,columns=["Elemento","Localización o explicación"]),"Tabla 5 · Recurso seleccionado: "+TIPO_RECURSO+" "+str(NUMERO_RECURSO)+" · Informe v0.28,16/11/2026")
# A compact full index avoids having to guess the other resource numbers.
indice=RECURSOS[["Tipo","Número","Página PDF","Título"]].copy()
indice["Abrir PDF"]=[enlace(PDF,"Página "+str(p),"#page="+str(p)) for p in indice["Página PDF"]]
mostrar(indice,"Tabla 6 · Índice completo de 43 tablas y 20 figuras · Informe v0.28,16/11/2026 · Páginas del PDF actual")


Elemento,Localización o explicación
Recurso,"Tabla 24 · Consumo y ahorro identificados para 2,5 millones de visitas en 2028"
PDF entregado y página,Informe v0.28 · página 51
Archivo del recurso,docs/informe_energetico/tablas/vigente/balance.tex
Sección del informe,Balance anual identificado
Unidades,data/gold/anexos/diccionario_columnas.csv; localizar las columnas de los anexos relacionados
Fuentes e hipótesis,config/parametros_modelo.xlsx
Referencias citadas en el informe,"calculo,copert,fta"
Entradas de aplicación documentadas,data/silver/operacion/oferta_vigente | data/silver/energia/lanzaderas/referencias_por_servicio | data/silver/energia/coche_hasta_cabeceras/reasignaciones | data/silver/energia/excursiones/referencias_por_origen.csv | data/silver/energia/titsa
Código de cálculo documentado,src/movilidad_teide/flujos/ordenacion_vigente.py:137::reproducir_ordenacion | src/movilidad_teide/flujos/ordenacion_vigente.py:358::_anualizar_politicas | src/movilidad_teide/informe/anexos_vigentes.py:71::generar_anexos_vigentes
Productor o reutilización del recurso,src/movilidad_teide/informe/vigente.py:31::preparar_tablas


Tipo,Número,Página PDF,Título,Abrir PDF
Figura,1,7,Cabeceras y eje interior de la propuesta de movilidad para 2028,Página 7
Figura,2,9,Cómo se obtiene el ahorro para las mismas visitas de 2028,Página 9
Figura,3,12,"Qué transporte atiende las mismas 2,5 millones de visitas en 2028",Página 12
Figura,4,14,Dos hipótesis sobre el tipo de coche utilizado en 2028,Página 14
Figura,5,21,Ocho orígenes de ejemplo y sus cabeceras habituales de cálculo,Página 21
Figura,6,24,Eje interior y destinos de referencia de la propuesta para 2028,Página 24
Figura,7,26,Recorridos directos de coche utilizados para planificar 2028,Página 26
Figura,8,27,Recorridos habituales en coche hasta las cabeceras: planificación de 2028,Página 27
Figura,9,32,Comprobaciones necesarias para confirmar una jornada de transporte,Página 32
Figura,10,33,"Cómo se enlazan cabeceras, visita interior y regreso",Página 33


## 7. Interpretar un anexo y sus columnas

La tabla 7 localiza el anexo elegido y conserva su fuente, transformación, productor y límite. La tabla 8 muestra **todas sus columnas**, con unidades, periodo, significado y tratamiento de ausencias. Los archivos de datos y sus diccionarios no se modifican al inspeccionarlos.

El cero de un servicio sustituido o de plazas comerciales en un vacío es una cantidad explícita. Un consumo común no cuantificado sigue siendo un dato ausente, no consumo cero. También se distinguen visitas, embarques, expediciones y vehículos físicos. Las dos asignaciones de turismos representan las mismas visitas; las bandas no autorizan sumar alternativas ni restar extremos de cuentas distintas.

**Tablas 7–8. Procedencia y diccionario del anexo seleccionado de la referencia del Teide, 2028.** Fuentes: matrices de anexos y columnas, vinculadas con las cabeceras de los archivos entregados. La comprobación de cabeceras y relaciones no equivale a verificar de forma independiente cada valor del documento.


In [9]:
seleccion=ANEXOS.loc[ANEXOS.Anexo.map(lambda x:Path(x).name).eq(ANEXO_LECTURA)]
if len(seleccion)!=1:raise ValueError("El anexo seleccionado no existe o no tiene correspondencia única.")
ANEXO=seleccion.iloc[0]
campos=[(k,enlace(v) if k=="Anexo" else html.escape(str(v))) for k,v in ANEXO.items()]
mostrar(pd.DataFrame(campos,columns=["Elemento","Descripción o vínculo"]),"Tabla 7 · Procedencia del anexo "+ANEXO_LECTURA+" · Referencia Teide 2028")
diccionario=COLUMNAS.loc[COLUMNAS.Archivo.eq(ANEXO_LECTURA),["Columna","Tipo","Unidad","Periodo","Significado","Ausencias","Fuente y cálculo"]]
if diccionario.empty:raise ValueError("El anexo no tiene definiciones de columnas.")
mostrar(diccionario,"Tabla 8 · Diccionario íntegro de "+ANEXO_LECTURA+" · Referencia Teide 2028")


Elemento,Descripción o vínculo
Anexo,data/gold/anexos/balance_anual.csv
Resultado,Consumos sin/con y ahorro anual
Etapas,Balance
Ubicación en informe,"Resultados anuales, diarios y por régimen"
Fuentes e hipótesis,config/parametros_modelo.xlsx
Entradas identificadas,data/silver/operacion/oferta_vigente | data/silver/energia/lanzaderas/referencias_por_servicio | data/silver/energia/coche_hasta_cabeceras/reasignaciones | data/silver/energia/excursiones/referencias_por_origen.csv | data/silver/energia/titsa
Código,src/movilidad_teide/flujos/ordenacion_vigente.py:137::reproducir_ordenacion | src/movilidad_teide/flujos/ordenacion_vigente.py:358::_anualizar_politicas | src/movilidad_teide/informe/anexos_vigentes.py:71::generar_anexos_vigentes
Productor,src/movilidad_teide/informe/anexos_vigentes.py:71::generar_anexos_vigentes
Transformación,"Exportación o agregación en la generación de recursos; Sumar actividad por fecha y asignación, añadir posicionamiento por activación, restar sin menos con dentro de cada asignación; convertir energía y dividir por las mismas visitas del periodo."
Unidades,manifiestos/trazabilidad_columnas.csv; filtrar por este archivo


Columna,Tipo,Unidad,Periodo,Significado,Ausencias,Fuente y cálculo
Asignación de turismos,Texto,categoría,2028,"Tipo de coche de España no Tenerife; Tenerife usa particular y extranjeros, alquiler en ambas asignaciones.",No admitida,"Configuración efectiva, calendario y circulaciones programadas; consumo obtenido con factores de ejecuciones programáticas de COPERT y agregación de la actividad. Trazabilidad: manifiestos/trazabilidad_anexos.csv, archivo balance_anual.csv; manifiestos/trazabilidad_columnas.csv, archivo y columna cuando corresponda. La preparación se detalla en manifiestos/trazabilidad_preparacion_balance.csv."
Tratamiento energético,Texto,categoría,2028,Consumo en caliente o contraste FTA aplicado una vez solo a lanzaderas y posicionamiento.,No admitida,"Configuración efectiva, calendario y circulaciones programadas; consumo obtenido con factores de ejecuciones programáticas de COPERT y agregación de la actividad. Trazabilidad: manifiestos/trazabilidad_anexos.csv, archivo balance_anual.csv; manifiestos/trazabilidad_columnas.csv, archivo y columna cuando corresponda. La preparación se detalla en manifiestos/trazabilidad_preparacion_balance.csv."
Visitas totales,Número entero,visitas-persona,2028,Entradas de personas del ámbito de la fila; regreso e interior no añaden visitas.,No admitida,"Configuración efectiva, calendario y circulaciones programadas; consumo obtenido con factores de ejecuciones programáticas de COPERT y agregación de la actividad. Trazabilidad: manifiestos/trazabilidad_anexos.csv, archivo balance_anual.csv; manifiestos/trazabilidad_columnas.csv, archivo y columna cuando corresponda. La preparación se detalla en manifiestos/trazabilidad_preparacion_balance.csv."
Sin (MWh),Número decimal,MWh,2028,"Energía identificada sin del periodo/ámbito de la fila; extremos técnicos dentro de la misma asignación, no intervalos de confianza.",No admitida,"Configuración efectiva, calendario y circulaciones programadas; consumo obtenido con factores de ejecuciones programáticas de COPERT y agregación de la actividad. Trazabilidad: manifiestos/trazabilidad_anexos.csv, archivo balance_anual.csv; manifiestos/trazabilidad_columnas.csv, archivo y columna cuando corresponda. La preparación se detalla en manifiestos/trazabilidad_preparacion_balance.csv."
Sin (tep),Número decimal,tep,2028,"Energía identificada sin del periodo/ámbito de la fila; extremos técnicos dentro de la misma asignación, no intervalos de confianza.",No admitida,"Configuración efectiva, calendario y circulaciones programadas; consumo obtenido con factores de ejecuciones programáticas de COPERT y agregación de la actividad. Trazabilidad: manifiestos/trazabilidad_anexos.csv, archivo balance_anual.csv; manifiestos/trazabilidad_columnas.csv, archivo y columna cuando corresponda. La preparación se detalla en manifiestos/trazabilidad_preparacion_balance.csv."
Sin (MWh/visita),Número decimal,MWh/visita,2028,"Energía identificada sin del periodo/ámbito de la fila; extremos técnicos dentro de la misma asignación, no intervalos de confianza.",No admitida,"Configuración efectiva, calendario y circulaciones programadas; consumo obtenido con factores de ejecuciones programáticas de COPERT y agregación de la actividad. Trazabilidad: manifiestos/trazabilidad_anexos.csv, archivo balance_anual.csv; manifiestos/trazabilidad_columnas.csv, archivo y columna cuando corresponda. La preparación se detalla en manifiestos/trazabilidad_preparacion_balance.csv."
Sin (tep/visita),Número decimal,tep/visita,2028,"Energía identificada sin del periodo/ámbito de la fila; extremos técnicos dentro de la misma asignación, no intervalos de confianza.",No admitida,"Configuración efectiva, calendario y circulaciones programadas; consumo obtenido con factores de ejecuciones programáticas de COPERT y agregación de la actividad. Trazabilidad: manifiestos/trazabilidad_anexos.csv, archivo balance_anual.csv; manifiestos/trazabilidad_columnas.csv, archivo y columna cuando corresponda. La preparac

## 8. Hipótesis, fichas de cálculo y bibliografía

El Excel sigue siendo el **único editor de hipótesis**. La matriz identifica 31 valores, unidades y celdas; el registro impide continuar si el libro o sus condiciones no corresponden a la ejecución. Las funciones están en el código común. El índice de referencia y los CSV de consulta no son editores alternativos.

Las **ocho fichas metodológicas** explican demanda, recorridos, vehículos, energía, balance y entrega, con sus funciones y salidas. La tabla 9 permite localizar sus archivos sin repetir las instrucciones de ejecución en cada ficha. Las 22 referencias bibliográficas distinguen fuentes locales, hipótesis y referencias externas; su disponibilidad y sus límites se consultan en la matriz bibliográfica.

**Tabla 9. Ocho fichas de comprobación metodológica de la entrega v0.28.** Fuente: matriz de fichas. Las funciones y las correspondencias están comprobadas; el acceso a los documentos depende de la modalidad de entrega.


In [10]:
mostrar(FICHAS[["Ficha","Archivo","Código y funciones","Salidas","Estado"]],"Tabla 9 · Fichas metodológicas y código asociado · Informe v0.28,16/11/2026")
display(Markdown("Hipótesis: [31 condiciones y celdas del Excel](../manifiestos/trazabilidad_hipotesis.csv). Bibliografía: [22 referencias y disponibilidad](../manifiestos/fuentes_bibliograficas.csv). Código actual: [mapa de interfaces](../manifiestos/mapa_recorrido_codigo.csv)."))


Ficha,Archivo,Código y funciones,Salidas,Estado
"Demanda, aforos y calendario",docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/demanda/calendario_anual.py:18::crear_calendario | src/movilidad_teide/demanda/calendario_anual.py:30::construir_perfiles_anuales | src/movilidad_teide/demanda/calendario_anual.py:86::aplicar_cupo_global,data/gold/anexos/visitas_modos_colectivos.csv | data/gold/anexos/calendario_diario.csv | data/gold/anexos/parametros_vigentes.csv,Rutas comunes y prefijo Gold explícitos; funciones verificadas en la edición alineada
Viajeros de TITSA y servicios de acceso,docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/demanda/titsa.py:88::leer_demanda_publica_titsa | src/movilidad_teide/demanda/titsa.py:9::clasificar_pasajeros_titsa,data/gold/anexos/visitas_modos_colectivos.csv | data/gold/anexos/energia_componentes.csv | data/gold/anexos/visitas_origen_colectivo.csv | data/gold/anexos/titsa_recorridos.csv | data/gold/anexos/titsa_segmentos.csv | data/gold/anexos/titsa_factores.csv,Rutas comunes y prefijo Gold explícitos; funciones verificadas en la edición alineada
"Orígenes, geometrías y cabeceras",docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/demanda/cabeceras_alternativas.py:12::leer_catalogo | src/movilidad_teide/demanda/cabeceras_alternativas.py:40::trasladar_reservas,data/gold/anexos/recorridos_origenes.csv | data/gold/anexos/reasignacion_cabeceras.csv | data/gold/anexos/pendientes_recorridos.csv,Rutas comunes y prefijo Gold explícitos; funciones verificadas en la edición alineada
"Excursiones, acceso TITSA y consumos comunes",docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/flujos/ordenacion_vigente.py:241::_preparar_factores | src/movilidad_teide/flujos/ordenacion_vigente.py:308::_valorar_programa,data/gold/anexos/energia_componentes.csv | data/gold/anexos/balance_anual.csv | data/gold/anexos/sensibilidad_excursiones.csv,Rutas comunes y prefijo Gold explícitos; funciones verificadas en la edición alineada
"Reservas, circulaciones y vehículos físicos",docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/operacion/calendario_vehiculos.py:6::materializar_calendario,data/gold/anexos/movimientos_lanzaderas.csv | data/gold/anexos/reservas_visita.csv | data/gold/anexos/disponibilidad_unidades.csv | data/gold/anexos/programacion_anual.csv,Rutas comunes y prefijo Gold explícitos; funciones verificadas en la edición alineada
"Parque, factores y ajuste de pendiente",docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/energia/pendiente.py:211::energia_pendiente_por_recorrido | src/movilidad_teide/energia/pendiente.py:273::aplicar_correcciones_a_actividad,data/gold/anexos/factores_turismos_utilizados.csv | data/gold/anexos/parametros_fisicos.csv | data/gold/anexos/sensibilidad_fisica.csv,Rutas comunes y prefijo Gold explícitos; funciones verificadas en la edición alineada
"Balance anual, regímenes y sensibilidad",docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/flujos/ordenacion_vigente.py:137::reproducir_ordenacion | src/movilidad_teide/flujos/ordenacion_vigente.py:358::_anualizar_politicas | src/movilidad_teide/flujos/ordenacion_vigente.py:446::_anualizar_sensibilidad,data/gold/anexos/balance_anual.csv | data/gold/anexos/balance_regimenes.csv | data/gold/anexos/energia_diaria.csv | data/gold/anexos/sensibilidad_conjunta.csv | data/gold/anexos/umbrales_condicionados.csv,Rutas comunes y prefijo Gold explícitos; funciones verificadas en la edición alineada
Reproducción y preparación de la entrega,docs/informe_energetico/tablas/vigente/guia_calculos_completa.tex,src/movilidad_teide/informe/integracion.py:92::preparar_integracion | src/movilidad_teide/informe/integracion.py:65::analisis_algebraico,data/gold/anexos/indice_anexos.csv | data/gold/anexos/diccionario_columnas.c

Hipótesis: [31 condiciones y celdas del Excel](../manifiestos/trazabilidad_hipotesis.csv). Bibliografía: [22 referencias y disponibilidad](../manifiestos/fuentes_bibliograficas.csv). Código actual: [mapa de interfaces](../manifiestos/mapa_recorrido_codigo.csv).

## 9. Recursos del informe y evidencia disponible

El catálogo distingue recursos generados desde datos, recursos físicos reutilizados y una tabla de parámetros integrada en el maestro. La tabla 10 describe esos papeles. Copiar una figura no equivale a reconstruirla desde el terreno.

Las evidencias editoriales conservan la comparación de la edición aceptada, incluidas las imágenes de trece PDF con diferente huella binaria y el mismo dibujo a 120 puntos por pulgada. Son pruebas fechadas, no otro renderizado realizado por este cuaderno.

La celda coteja las huellas cuando los documentos están distribuidos. Si están reservados, indica cuántos recursos quedan catalogados y deja claro que no está comprobando sus bytes. El cálculo y sus referencias numéricas continúan siendo obligatorios.

**Tabla 10. Tratamiento de los 63 recursos numerados del informe v0.28.** Fuente: matriz y evidencias editoriales. Se distinguen reproducción de datos, generación de recursos, reutilización y disponibilidad efectiva.


In [11]:
tratamiento=RECURSOS.groupby(["Tipo","Tratamiento del recurso"],sort=False).size().rename("Recursos").reset_index()
mostrar(tratamiento,"Tabla 10 · Recursos generados, reutilizados e integrados · Informe v0.28,16/11/2026")
EDITORIAL=leer_matriz("comparacion_recursos_editoriales")
VISUAL=leer_matriz("comparacion_visual_recursos")
COTEJADOS=0
for r in EDITORIAL.to_dict("records"):
    if POLITICA_DOCUMENTAL.excluye(r["Recurso"]):
        continue
    if hashlib.sha256((RAIZ/r["Recurso"]).read_bytes()).hexdigest()!=r["SHA256 entrega"]:
        raise ValueError("Cambió un recurso de la entrega: "+r["Recurso"])
    COTEJADOS+=1
print("Recursos documentales cotejados ahora:",COTEJADOS)
print("La verificación documental aplica la política de .gitignore; el catálogo conserva sus 63 recursos.")
print("Comparaciones visuales documentadas a 120 ppp, sin repetir renderizado ahora:",len(VISUAL))


Tipo,Tratamiento del recurso,Recursos
Figura,Recurso físico identificado que se copia de la entrega,6
Figura,Generado en destino nuevo,14
Tabla,Generado en destino nuevo,37
Tabla,Tabla física copiada de la entrega y revisada editorialmente,5
Tabla,"Tabla integrada en el maestro; parámetros contrastados, sin productor de filas independiente",1


Recursos documentales cotejados ahora: 0
La verificación documental aplica la política de .gitignore; el catálogo conserva sus 63 recursos.
Comparaciones visuales documentadas a 120 ppp, sin repetir renderizado ahora: 20


## 10. Alcance de las comprobaciones

La secuencia 00–07 permite seguir datos, hipótesis, preparaciones, servicio, factores, balance y comprobaciones con sesiones nuevas de Python. La aceptación distingue el recorrido que consulta factores distribuidos del que utiliza las ejecuciones reales de COPERT de 05. Las preparaciones ya verificadas se reutilizan con sus huellas; no se atribuye una reconstrucción nueva desde Bronze a una simple lectura o comparación.

Las 33 comparaciones del balance incluyen 16 productos anuales y 17 interfaces físicas/de coeficientes. La vista física anual adicional de 06 tiene 24 filas: su equivalencia separada está documentada en la [aceptación de 06](../manifiestos/comprobacion_entrega_final.json), junto con sus entradas y código. No se cuenta como otras 24 comparaciones ejecutadas por esta etapa 07.

La comprobación conserva los límites del estudio: energía identificada con partidas comunes aún no cuantificadas; bandas técnicas sin probabilidades; programación de planificación y necesidad de confirmar medios y reservas; terreno y rasante distintos; y diferencia de cobertura entre los microdatos del parque y el contraste municipal. Los datos, hipótesis y tolerancias no se ajustan para ocultar estas limitaciones.

El [informe v0.28](../docs/informe_energetico/INFORME_ENERGETICO_v0.28_2026-11-16.pdf) y sus anexos se relacionan con los datos y productores mediante las matrices de trazabilidad. La [guía de vigencia y procedencia](../docs/vigencia_y_procedencia.md) explica el papel de las fuentes, preparaciones y referencias que se consultan en este recorrido.

La [guía de reproducción](../docs/reproduccion.md) reúne las instrucciones de ejecución. La [guía de entrega](../docs/entrega_publica.md) identifica los paquetes y sus controles. El acta que acompaña a cada entrega distingue sus comprobaciones documentales y numéricas, sin equiparar la identidad de archivos con una validación operativa.
